### Required Discussion 19:1: Building a Recommender System with SURPRISE

This discussion focuses on exploring additional algorithms with the `Suprise` library to generate recommendations.  Your goal is to identify the optimal algorithm by minimizing the mean squared error using cross validation. You are also going to select a dataset to use from [grouplens](https://grouplens.org/datasets/movielens/) example datasets.  

To begin, head over to [grouplens](https://grouplens.org/datasets/movielens/) and examine the different datasets available.  Choose one so that it is easy to create the data as expected in `Surprise` with user, item, and rating information.  Then, compare the performance of at least the `KNNBasic`, `SVD`, `NMF`, `SlopeOne`, and `CoClustering` algorithms to build your recommendations.  For more information on the algorithms see the documentation for the algorithm package [here](https://surprise.readthedocs.io/en/stable/prediction_algorithms_package.html).

Share the results of your investigation and include the results of your cross validation and a basic description of your dataset with your peers.


In [ ]:
# Install once in a separate notebook cell if necessary:
# %pip install "numpy<2" pandas matplotlib scikit-surprise
# Restart the Jupyter kernel after installation if the import fails.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from surprise import Dataset, Reader, SVD, NMF, KNNBasic, SlopeOne, CoClustering
from surprise.model_selection import cross_validate, KFold
from IPython.display import display

In [ ]:
# Use the official, fixed MovieLens 100K dataset.
# Surprise downloads it once if needed; respond 'Y' when prompted.
# The user/item/rating format is provided by the Surprise dataset loader.
data = Dataset.load_builtin('ml-100k', prompt=False)
raw_ratings = data.raw_ratings
ratings_df = pd.DataFrame(raw_ratings, columns=['user_id', 'movie_id', 'rating', 'timestamp'])
ratings_df['rating'] = ratings_df['rating'].astype(float)
print(f'Ratings: {len(ratings_df):,}')
print(f'Users: {ratings_df.user_id.nunique():,}')
print(f'Movies: {ratings_df.movie_id.nunique():,}')
print(f'Rating range: {ratings_df.rating.min():.0f} to {ratings_df.rating.max():.0f}')
display(ratings_df.head())
assert len(ratings_df) == 100_000, 'Expected MovieLens 100K, not Latest Small.'

In [ ]:
# Same seeded 5-fold partition for every algorithm.
# Cross-validation reports RMSE and MAE; per-fold MSE = RMSE ** 2.
SEED = 42
algorithms = {
    'KNNBasic': KNNBasic(verbose=False),
    'SVD': SVD(random_state=SEED),
    'NMF': NMF(random_state=SEED),
    'SlopeOne': SlopeOne(),
    'CoClustering': CoClustering(random_state=SEED),
}
results = []
fold_details = []
for name, algorithm in algorithms.items():
    print(f'Evaluating {name} ...')
    cv = KFold(n_splits=5, random_state=SEED, shuffle=True)
    scores = cross_validate(algorithm, data, measures=['RMSE', 'MAE'], cv=cv, n_jobs=1, verbose=False)
    mse_folds = np.square(scores['test_rmse'])
    results.append({'Algorithm': name, 'Mean MSE': mse_folds.mean(),
                    'Std MSE': mse_folds.std(ddof=1),
                    'Mean RMSE': scores['test_rmse'].mean(),
                    'Mean MAE': scores['test_mae'].mean(),
                    'Mean Fit Time (s)': np.mean(scores['fit_time'])})
    for fold, (mse, rmse, mae) in enumerate(zip(mse_folds, scores['test_rmse'], scores['test_mae']), 1):
        fold_details.append({'Algorithm': name, 'Fold': fold, 'MSE': mse, 'RMSE': rmse, 'MAE': mae})
results_df = pd.DataFrame(results).sort_values('Mean MSE').reset_index(drop=True)
folds_df = pd.DataFrame(fold_details)
print('Cross-validation summary (lower is better):')
display(results_df.round(4))
print('Per-fold scores:')
display(folds_df.round(4))

In [ ]:
# Plot model comparison and save machine-readable results.
project_root = Path.cwd()
if not (project_root / 'README.md').exists():
    project_root = project_root.parent
assert (project_root / 'README.md').exists(), 'Run from project root or notebooks/ folder.'
figures_dir = project_root / 'outputs' / 'figures'
reports_dir = project_root / 'outputs' / 'reports'
figures_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(reports_dir / 'cross_validation_summary.csv', index=False)
folds_df.to_csv(reports_dir / 'cross_validation_folds.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(results_df['Algorithm'], results_df['Mean MSE'], yerr=results_df['Std MSE'], capsize=4)
ax.set(title='MovieLens 100K: 5-Fold Cross-Validation MSE', ylabel='Mean Squared Error (lower is better)', xlabel='Algorithm')
ax.set_ylim(bottom=0)
fig.tight_layout()
fig.savefig(figures_dir / 'model_comparison.png', dpi=160)
plt.show()
print(f"Best algorithm by mean MSE: {results_df.iloc[0]['Algorithm']} ({results_df.iloc[0]['Mean MSE']:.4f})")

In [ ]:
# Generate a <=200-word discussion post using the actual measured results.
best = results_df.iloc[0]
score_lines = '\n'.join(f"- {r['Algorithm']}: MSE={r['Mean MSE']:.4f}, RMSE={r['Mean RMSE']:.4f}, MAE={r['Mean MAE']:.4f}" for _, r in results_df.iterrows())
post = f"""For Required Discussion 19.1, I used the original MovieLens 100K dataset from GroupLens, containing 100,000 movie ratings from 943 users across 1,682 movies. Ratings range from 1 to 5 stars. Each record contains a user ID, movie ID, and rating.

I compared KNNBasic, SVD, NMF, SlopeOne, and CoClustering using the SURPRISE library and the same seeded 5-fold cross-validation setup. The primary selection metric was mean squared error (MSE), calculated as the square of each fold's RMSE before averaging.

Cross-validation results:
{score_lines}

{best['Algorithm']} achieved the lowest mean MSE ({best['Mean MSE']:.4f}) and was therefore selected as the best model among those tested. Lower MSE indicates more accurate predictions of users' movie ratings on unseen folds. This exercise demonstrates how cross-validation can help compare recommendation methods consistently. A future extension would evaluate top-N recommendation quality, not only rating prediction error."""
word_count = len(post.split())
print(f'Discussion post word count: {word_count} (limit: 200)')
assert word_count <= 200
print('\n' + post)
(reports_dir / 'discussion_post.txt').write_text(post, encoding='utf-8')